In [1]:
import pandas as pd
import os

qlist_path = "stock23_test_qlist.json"
clist_path = "stock23_test_clist.json"

print("Parsing JSON Lines format...")

# 2. Data Parsing (JSONL mode)
try:
    df_queries = pd.read_json(qlist_path, lines=True)
    df_candidates = pd.read_json(clist_path, lines=True)
    print("Parsing successful.")
except Exception as e:
    print(f"FATAL: Parsing failed: {e}")

# 3. Structural Inspection
print(f"\nQueries Shape: {df_queries.shape}")
print(f"Candidates Shape: {df_candidates.shape}")

print("\n--- Query Null Value Count ---")
print(df_queries.isnull().sum())

# 4. Dimensionality Validation (5-Day Constraint on Queries)
expected_columns = ['query_stock', 'query_date', 'recent_date_list', 'adjusted_close_list']
missing_cols = [col for col in expected_columns if col not in df_queries.columns]

if missing_cols:
    print(f"\nFATAL: Missing expected serialization columns: {missing_cols}")
else:
    def validate_5_day_window(row):
        return len(row['recent_date_list']) == 5 and len(row['adjusted_close_list']) == 5

    df_queries['valid_window'] = df_queries.apply(validate_5_day_window, axis=1)
    invalid_rows = (~df_queries['valid_window']).sum()
    
    print("\n--- Dimensionality Check ---")
    print(f"Sequences failing 5-day constraint: {invalid_rows}")

# 5. Canonical State Output
print("\n--- Sample Query Record ---")
print(df_queries.drop(columns=['valid_window'], errors='ignore').head(1).T)

Parsing JSON Lines format...
Parsing successful.

Queries Shape: (4128, 39)
Candidates Shape: (404736, 39)

--- Query Null Value Count ---
data_index                                 0
query_stock                                0
query_date                                 0
movement                                   0
date_list                                  0
adj_close_list                             0
open_list                                  0
high_list                                  0
low_list                                   0
close_list                                 0
volume_list                                0
movement_list                              0
MACD_Histogram_list                        0
macd_crossover_list                        0
bollinger_bands_list                       0
exceeding_upper_list                       0
exceeding_lower_list                       0
overbought_and_oversold_conditions_list    0
kdj_crossover_list                         0
Return

In [2]:
import json

# 1. Standardize Nomenclature
rename_map = {
    'date_list': 'recent_date_list',
    'adj_close_list': 'adjusted_close_list'
}
df_queries.rename(columns=rename_map, inplace=True, errors='ignore')
df_candidates.rename(columns=rename_map, inplace=True, errors='ignore')

# 2. Isolate a Single Query
query_row = df_queries.iloc[0] 

# 3. Create a Mini Candidate Pool for Local Testing
candidate_rows = df_candidates.head(1000)

# Extract safely regardless of prefix
q_stock = query_row.get('query_stock', query_row.get('stock', 'UNKNOWN'))
q_date = query_row.get('query_date', query_row.get('date', 'UNKNOWN'))

print(f"Target Query Stock: {q_stock} on {q_date}")
print(f"Constructed search pool of {len(candidate_rows)} historical candidates.")

# 4. Construct Query JSON String
query_dict = {
    "query_stock": q_stock,
    "query_date": q_date,
    "recent_date_list": query_row['recent_date_list'],
    "adjusted_close_list": query_row['adjusted_close_list']
}
query_str = json.dumps(query_dict)

# 5. Construct Candidate Pool JSON Strings
candidate_pool = []
for _, row in candidate_rows.iterrows():
    cand_dict = row.drop(labels=['data_index'], errors='ignore').to_dict()
    
    # Safely extract and normalize candidate keys
    stock_val = cand_dict.pop('candidate_stock', cand_dict.pop('query_stock', cand_dict.pop('stock', None)))
    date_val = cand_dict.pop('candidate_date', cand_dict.pop('query_date', cand_dict.pop('date', None)))
    mov_val = cand_dict.pop('candidate_movement', cand_dict.pop('movement', None))
    
    cand_dict['candidate_stock'] = stock_val
    cand_dict['candidate_date'] = date_val
    cand_dict['candidate_movement'] = mov_val
    
    candidate_pool.append(json.dumps(cand_dict))

print("\n--- Serialized Query String ---")
print(query_str)

print("\n--- Serialized Candidate String (Sample) ---")
print(candidate_pool[0][:300] + "...")

Target Query Stock: MRVL on 2023-01-03
Constructed search pool of 1000 historical candidates.

--- Serialized Query String ---
{"query_stock": "MRVL", "query_date": "2023-01-03", "recent_date_list": ["2022-12-23", "2022-12-27", "2022-12-28", "2022-12-29", "2022-12-30"], "adjusted_close_list": [36.956, 35.8232, 35.1574, 36.5287, 36.807]}

--- Serialized Candidate String (Sample) ---
{"recent_date_list": ["2022-01-03", "2022-01-04", "2022-01-05", "2022-01-06", "2022-01-07"], "adjusted_close_list": [88.4925, 87.4436, 83.2381, 84.6937, 82.2974], "open_list": NaN, "high_list": NaN, "low_list": NaN, "close_list": NaN, "volume_list": NaN, "movement_list": NaN, "MACD_Histogram_list": ...


In [3]:
import pandas as pd
import json

# Replace NaN with None for JSON compliance
candidate_rows_clean = candidate_rows.where(pd.notnull(candidate_rows), None)

candidate_pool = []
for _, row in candidate_rows_clean.iterrows():
    cand_dict = row.drop(labels=['data_index'], errors='ignore').to_dict()
    
    stock_val = cand_dict.pop('candidate_stock', cand_dict.pop('query_stock', cand_dict.pop('stock', None)))
    date_val = cand_dict.pop('candidate_date', cand_dict.pop('query_date', cand_dict.pop('date', None)))
    mov_val = cand_dict.pop('candidate_movement', cand_dict.pop('movement', None))
    
    cand_dict['candidate_stock'] = stock_val
    cand_dict['candidate_date'] = date_val
    cand_dict['candidate_movement'] = mov_val
    
    candidate_pool.append(json.dumps(cand_dict))

sample_output = candidate_pool[0]
assert "NaN" not in sample_output, "Sanitization failed: NaN persists."
print("Sanitization complete. Sample string verified compliant.")
print(sample_output[:300] + "...")

Sanitization complete. Sample string verified compliant.
{"recent_date_list": ["2022-01-03", "2022-01-04", "2022-01-05", "2022-01-06", "2022-01-07"], "adjusted_close_list": [88.4925, 87.4436, 83.2381, 84.6937, 82.2974], "open_list": null, "high_list": null, "low_list": null, "close_list": null, "volume_list": null, "movement_list": null, "MACD_Histogram_l...


In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# 1. Hardware Allocation
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Allocating FinSeer to {device}...")

retriever_tokenizer = AutoTokenizer.from_pretrained("TheFinAI/FinSeer")
retriever_model = AutoModel.from_pretrained("TheFinAI/FinSeer").to(device)
retriever_model.eval()

def get_embedding(text):
    # Tokenize and push to GPU
    inputs = retriever_tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = retriever_model(**inputs)
    # Extract the CLS token (index 0) as the sequence representation
    return outputs.last_hidden_state[:, 0, :]

# 2. Query Vectorization
print("Vectorizing query sequence...")
query_emb = get_embedding(query_str)

# 3. Candidate Vectorization & Scoring
print(f"Vectorizing {len(candidate_pool)} candidates and computing cosine similarity...")
scores = []

for cand_str in candidate_pool:
    cand_emb = get_embedding(cand_str)
    # Calculate similarity metric
    sim = F.cosine_similarity(query_emb, cand_emb).item()
    scores.append((sim, cand_str))

# Sort by highest similarity
scores.sort(key=lambda x: x[0], reverse=True)
top_score = scores[0][0]
top_candidate = scores[0][1]

# 4. Result Extraction
print("\n--- Retrieval Results ---")
print(f"Maximum Cosine Similarity Score: {top_score:.4f}")
print(f"Top Context Sequence (Truncated):\n{top_candidate[:500]}...")

Allocating FinSeer to cuda...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: TheFinAI/FinSeer
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Vectorizing query sequence...
Vectorizing 1000 candidates and computing cosine similarity...

--- Retrieval Results ---
Maximum Cosine Similarity Score: 0.9780
Top Context Sequence (Truncated):
{"recent_date_list": ["2022-01-31", "2022-02-01", "2022-02-02", "2022-02-03", "2022-02-04"], "adjusted_close_list": [70.7019, 71.7119, 73.3557, 67.959, 70.4048], "open_list": null, "high_list": null, "low_list": null, "close_list": null, "volume_list": null, "movement_list": null, "MACD_Histogram_list": null, "macd_crossover_list": null, "bollinger_bands_list": null, "exceeding_upper_list": null, "exceeding_lower_list": null, "overbought_and_oversold_conditions_list": null, "kdj_crossover_list":...


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Hardware Allocation
print("Allocating StockLLM to VRAM (fp16)...")
gen_tokenizer = AutoTokenizer.from_pretrained("TheFinAI/StockLLM")
gen_model = AutoModelForCausalLM.from_pretrained(
    "TheFinAI/StockLLM", 
    device_map="auto", 
    torch_dtype=torch.float16
)
gen_model.eval()

# 2. Prompt Formatting
raw_prompt = f"""Based on the following information, predict stock movement by filling in the [blank] with 'rise' or 'fall'.
Just fill in the blank, do not explain.
These are sequences that may affect this stock's price recently: {top_candidate}
This is the query sequence: {query_str}
Query: On {query_dict['query_date']}, the movement of {query_dict['query_stock']} is [blank]."""

messages = [
    {"role": "user", "content": raw_prompt}
]

# Apply LLaMA 3 syntax to prevent EOS truncation
formatted_prompt = gen_tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True 
)

inputs = gen_tokenizer(formatted_prompt, return_tensors="pt").to(device)

# 3. Deterministic Inference
print("Executing inference...")
with torch.no_grad():
    outputs = gen_model.generate(
        **inputs, 
        max_new_tokens=10, 
        do_sample=False, 
        pad_token_id=gen_tokenizer.eos_token_id 
    )

# 4. Output Parsing
input_length = inputs['input_ids'].shape[1]
generated_tokens = outputs[0][input_length:]
prediction = gen_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("\n--- Model Prediction ---")
print(f"Target: {query_dict['query_stock']} on {query_dict['query_date']}")
print(f"Predicted State: {prediction}")

Allocating StockLLM to VRAM (fp16)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Executing inference...

--- Model Prediction ---
Target: MRVL on 2023-01-03
Predicted State: fall


In [6]:
# ------- TSLA Query Example ---------

In [22]:
# 1. Isolate the Ticker Universe
qualified_tickers = df_queries['query_stock'].unique().tolist()
qualified_tickers.sort()

print(f"Total Qualified Tickers in Query Dataset: {len(qualified_tickers)}")
print(f"Ticker Universe:\n{qualified_tickers}\n")

# 2. Isolate the Feature Set
qualified_features = df_queries.columns.tolist()

print(f"Total Qualified Features/Factors: {len(qualified_features)}")
print(f"Feature Array:\n{qualified_features}")

Total Qualified Tickers in Query Dataset: 24
Ticker Universe:
['MRVL', 'MS', 'MSCI', 'NDAQ', 'NKE', 'NVDA', 'ORLY', 'PANW', 'PEAK', 'PLTR', 'PNC', 'PNW', 'RY', 'SMCI', 'SNOW', 'SNPS', 'T', 'TGT', 'TSLA', 'UBER', 'ULTA', 'V', 'WBA', 'WDAY']

Total Qualified Features/Factors: 39
Feature Array:
['data_index', 'query_stock', 'query_date', 'movement', 'recent_date_list', 'adjusted_close_list', 'open_list', 'high_list', 'low_list', 'close_list', 'volume_list', 'movement_list', 'MACD_Histogram_list', 'macd_crossover_list', 'bollinger_bands_list', 'exceeding_upper_list', 'exceeding_lower_list', 'overbought_and_oversold_conditions_list', 'kdj_crossover_list', 'Returns_list', 'VWAP_list', 'alpha_smr_list', 'alpha_mom_list', 'alpha002_list', 'alpha006_list', 'alpha009_list', 'alpha012_list', 'alpha021_list', 'alpha023_list', 'alpha024_list', 'alpha028_list', 'alpha032_list', 'alpha041_list', 'alpha046_list', 'alpha049_list', 'alpha051_list', 'alpha053_list', 'alpha054_list', 'alpha101_list']


In [23]:
TARGET_TICKER = 'PLTR'

# Extract and sort available dates for the target ticker
valid_dates = df_queries[df_queries['query_stock'] == TARGET_TICKER]['query_date'].sort_values().tolist()

print(f"Total available query sequences for {TARGET_TICKER}: {len(valid_dates)}")
print(f"Valid query dates:\n{valid_dates}")

Total available query sequences for PLTR: 150
Valid query dates:
['2023-01-04', '2023-01-05', '2023-01-06', '2023-01-09', '2023-01-10', '2023-01-11', '2023-01-13', '2023-01-18', '2023-01-19', '2023-01-20', '2023-01-23', '2023-01-24', '2023-01-27', '2023-01-30', '2023-02-01', '2023-02-02', '2023-02-03', '2023-02-06', '2023-02-08', '2023-02-09', '2023-02-10', '2023-02-14', '2023-02-15', '2023-02-16', '2023-02-17', '2023-02-21', '2023-02-22', '2023-02-23', '2023-02-24', '2023-02-27', '2023-02-28', '2023-03-01', '2023-03-02', '2023-03-06', '2023-03-08', '2023-03-09', '2023-03-10', '2023-03-17', '2023-03-20', '2023-03-22', '2023-03-24', '2023-03-27', '2023-03-29', '2023-03-30', '2023-03-31', '2023-04-03', '2023-04-05', '2023-04-12', '2023-04-13', '2023-04-18', '2023-04-19', '2023-04-20', '2023-04-24', '2023-04-25', '2023-04-27', '2023-04-28', '2023-05-02', '2023-05-04', '2023-05-11', '2023-05-12', '2023-05-16', '2023-05-18', '2023-05-22', '2023-05-24', '2023-06-01', '2023-06-05', '2023-06-0

In [25]:
import json
import torch
import torch.nn.functional as F

# Define explicit query targets
TARGET_TICKER = 'PLTR'
TARGET_DATE = '2023-12-29'

# 1. Isolate Target Query via Compound Boolean Mask
target_queries = df_queries[(df_queries['query_stock'] == TARGET_TICKER) & (df_queries['query_date'] == TARGET_DATE)]

if target_queries.empty:
    raise ValueError(f"Sequence for {TARGET_TICKER} on {TARGET_DATE} not found in query dataset.")

target_row = target_queries.iloc[0]

q_stock = target_row.get('query_stock', target_row.get('stock'))
q_date = target_row.get('query_date', target_row.get('date'))

print(f"Target Isolated: {q_stock} on {q_date}")

query_dict = {
    "query_stock": q_stock,
    "query_date": q_date,
    "recent_date_list": target_row['recent_date_list'],
    "adjusted_close_list": target_row['adjusted_close_list']
}
query_str = json.dumps(query_dict)

# 2. Retriever Execution
print("Vectorizing explicit query sequence...")
query_emb = get_embedding(query_str)

scores = []
for cand_str in candidate_pool:
    cand_emb = get_embedding(cand_str)
    sim = F.cosine_similarity(query_emb, cand_emb).item()
    scores.append((sim, cand_str))

scores.sort(key=lambda x: x[0], reverse=True)
top_candidate = scores[0][1]
print(f"Top Context Similarity Score: {scores[0][0]:.4f}")

# 3. Generator Execution
raw_prompt = f"""Based on the following information, predict stock movement by filling in the [blank] with 'rise' or 'fall'.
Just fill in the blank, do not explain.
These are sequences that may affect this stock's price recently: {top_candidate}
This is the query sequence: {query_str}
Query: On {query_dict['query_date']}, the movement of {query_dict['query_stock']} is [blank]."""

messages = [{"role": "user", "content": raw_prompt}]
formatted_prompt = gen_tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

inputs = gen_tokenizer(formatted_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = gen_model.generate(
        **inputs, 
        max_new_tokens=10, 
        do_sample=False, 
        pad_token_id=gen_tokenizer.eos_token_id 
    )

input_length = inputs['input_ids'].shape[1]
prediction = gen_tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

# 4. State Output
print("\n--- Output State ---")
print(f"Target Vector: {q_stock} | {q_date}")
print(f"Predicted State: {prediction}")
print(f"Dataset Ground Truth: {target_row.get('movement', 'UNKNOWN')}")

Target Isolated: PLTR on 2023-12-29
Vectorizing explicit query sequence...
Top Context Similarity Score: 0.9522

--- Output State ---
Target Vector: PLTR | 2023-12-29
Predicted State: fall
Dataset Ground Truth: fall
